In [ ]:
import numpy as np
import matplotlib.pyplot as plt

```
# conventions visuelles fortes
# Pôle céleste → rouge
# Zénith → vert
# Objet → bleu
# Cercle horaire → bleu foncé
# Cercle vertical → vert foncé
# Horizon → orange
# Méridien → noir pointillé
```

In [ ]:
# -----------------------------
# Fonction pour tracer grand cercle
# -----------------------------
def great_circle(normal, n_points=200):
    normal = normal / np.linalg.norm(normal)

    # vecteur orthogonal
    if abs(normal[0]) < 0.9:
        ref = np.array([1, 0, 0])
    else:
        ref = np.array([0, 1, 0])

    v1 = np.cross(normal, ref)
    v1 /= np.linalg.norm(v1)

    v2 = np.cross(normal, v1)

    t = np.linspace(0, 2 * np.pi, n_points)
    circle = np.outer(np.cos(t), v1) + np.outer(np.sin(t), v2)

    return circle

In [ ]:
def filled_disk_old(normal, radius=1.0, n_r=30, n_theta=100):
    normal = normal / np.linalg.norm(normal)

    # base orthonormée du plan
    if abs(normal[0]) < 0.9:
        ref = np.array([1, 0, 0])
    else:
        ref = np.array([0, 1, 0])

    v1 = np.cross(normal, ref)
    v1 /= np.linalg.norm(v1)
    v2 = np.cross(normal, v1)

    r = np.linspace(0, radius, n_r)
    theta = np.linspace(0, 2 * np.pi, n_theta)

    R, T = np.meshgrid(r, theta)

    X = R * np.cos(T)
    Y = R * np.sin(T)

    disk = (
        np.outer(np.ones_like(theta), v1) * X[:, :, None] + np.outer(np.ones_like(theta), v2) * Y[:, :, None]
    )

    return disk

In [ ]:
def filled_disk(normal, radius=1.0, n_r=30, n_theta=100):
    normal = normal / np.linalg.norm(normal)

    # base orthonormée du plan
    if abs(normal[0]) < 0.9:
        ref = np.array([1, 0, 0])
    else:
        ref = np.array([0, 1, 0])

    v1 = np.cross(normal, ref)
    v1 /= np.linalg.norm(v1)
    v2 = np.cross(normal, v1)

    r = np.linspace(0, radius, n_r)
    theta = np.linspace(0, 2 * np.pi, n_theta)

    R, T = np.meshgrid(r, theta)

    # construction du disque (broadcast correct)
    X = R * np.cos(T)
    Y = R * np.sin(T)

    disk = X[..., None] * v1[None, None, :] + Y[..., None] * v2[None, None, :]

    return disk

In [ ]:
def draw_vec(v, label, color):
    ax.quiver(0, 0, 0, v[0], v[1], v[2], color=color, linewidth=2)
    ax.text(v[0] * 1.1, v[1] * 1.1, v[2] * 1.1, label, color=color, fontsize=10)

In [ ]:
def draw_point(v, label, color):
    ax.scatter(v[0], v[1], v[2], color=color, s=80)
    ax.text(v[0] * 1.1, v[1] * 1.1, v[2] * 1.1, label, color=color, fontsize=10)

In [ ]:
# -----------------------------
# Paramètres observateur / objet
# -----------------------------
lat = np.radians(45.0)  # latitude observateur
dec = np.radians(20.0)  # déclinaison objet
H = np.radians(45.0)  # angle horaire

# -----------------------------
# Vecteurs fondamentaux
# -----------------------------

# Pôle céleste (Nord)
P = np.array([0, 0, 1])

# Zénith (dépend de latitude)
Z = np.array([np.cos(lat), 0, np.sin(lat)])

# Objet (coordonnées équatoriales -> cartésien)
O = np.array([np.cos(dec) * np.cos(H), np.cos(dec) * np.sin(H), np.sin(dec)])

# -----------------------------
# Plans → normales
# -----------------------------

# Plan cercle horaire (objet + pôle)
n_hour = np.cross(P, O)

# Plan vertical (objet + zénith)
n_vert = np.cross(Z, O)


# -----------------------------
# Horizon = plan perpendiculaire au ZENITH
# -----------------------------
n_horizon = Z  # normal du plan horizon

# local meridian circle
n_meridian = np.cross(P, Z)

# -----------------------------
# Angle parallactique
# -----------------------------
q = np.arccos(np.dot(n_hour, n_vert) / (np.linalg.norm(n_hour) * np.linalg.norm(n_vert)))

print(f"Angle parallactique (deg): {np.degrees(q):.2f}")

In [ ]:
# -----------------------------
# Création sphère
# -----------------------------
u = np.linspace(0, 2 * np.pi, 100)
v = np.linspace(0, np.pi, 100)

x = np.outer(np.cos(u), np.sin(v))
y = np.outer(np.sin(u), np.sin(v))
z = np.outer(np.ones_like(u), np.cos(v))


circle_hour = great_circle(n_hour)
circle_vert = great_circle(n_vert)
circle_horizon = great_circle(n_horizon)
circle_meridian = great_circle(n_meridian)

# équateur céleste (plan ⟂ P)
circle_equator = great_circle(P)

disk_horizon = filled_disk(Z)


# projections
# projection de O sur le plan horizon
O_proj = O - np.dot(O, Z) * Z
O_proj /= np.linalg.norm(O_proj)

In [ ]:
# -----------------------------
# Points cardinaux sur l’horizon
# -----------------------------

# Nord = projection du pôle sur l’horizon
N = P - np.dot(P, Z) * Z
N /= np.linalg.norm(N)

# Est = Z × Nord
E = np.cross(Z, N)
E /= np.linalg.norm(E)

# Sud / Ouest
S = -N
W = -E

In [ ]:
# -----------------------------
# Base tangentielle au point O
# -----------------------------

# direction vers pôle (cercle horaire)
t_pole = P - np.dot(P, O) * O
t_pole /= np.linalg.norm(t_pole)

# direction vers zénith (cercle vertical)
t_zenith = Z - np.dot(Z, O) * O
t_zenith /= np.linalg.norm(t_zenith)

In [ ]:
# -----------------------------
# Plot
# -----------------------------
fig = plt.figure(figsize=(16, 16))
ax = fig.add_subplot(111, projection="3d")

# sphère
ax.plot_surface(x, y, z, alpha=0.1)


# horizon rempli (léger)
ax.plot_surface(
    disk_horizon[:, :, 0], disk_horizon[:, :, 1], disk_horizon[:, :, 2], alpha=0.2, color="orange", zorder=0
)

# cercle horizon
ax.plot(
    circle_horizon[:, 0],
    circle_horizon[:, 1],
    circle_horizon[:, 2],
    color="orange",
    linewidth=2,
    label="Horizon",
)


# grands cercles
ax.plot(circle_hour[:, 0], circle_hour[:, 1], circle_hour[:, 2], label="Cercle horaire", linewidth=2)

ax.plot(circle_vert[:, 0], circle_vert[:, 1], circle_vert[:, 2], label="Cercle vertical", linewidth=2)


# méridien local
ax.plot(
    circle_meridian[:, 0],
    circle_meridian[:, 1],
    circle_meridian[:, 2],
    linestyle="--",
    color="black",
    label="Méridien local",
)


# équateur céleste (plan ⟂ P)
circle_equator = great_circle(P)

ax.plot(
    circle_equator[:, 0],
    circle_equator[:, 1],
    circle_equator[:, 2],
    color="red",
    linestyle=":",
    label="Équateur céleste",
)

# objet
ax.scatter(O[0], O[1], O[2], color="blue", edgecolor="black", s=180, marker="*", label="Objet", zorder=5)


# projection objet sur horizon
ax.scatter(O_proj[0], O_proj[1], O_proj[2], color="purple", s=50, label="Proj horizon")

# ligne verticale (objet → horizon)
ax.plot([O[0], O_proj[0]], [O[1], O_proj[1]], [O[2], O_proj[2]], linestyle=":", color="purple")


# vecteurs
# def draw_vec(v, label, color):
#    ax.quiver(0,0,0, v[0],v[1],v[2], color=color, length=1.0)
#    ax.text(v[0], v[1], v[2], label, color=color)

draw_vec(P, "Pole", "red")
draw_vec(Z, "Zenith", "green")
ax.plot([0, Z[0]], [0, Z[1]], [0, Z[2]], color="green", linewidth=2)


draw_vec(O, "Objet", "blue")


# tracer les points cardinaux
draw_point(N, "N", "black")
draw_point(S, "S", "black")
draw_point(E, "E", "black")
draw_point(W, "W", "black")

ax.quiver(0, 0, 0, N[0], N[1], N[2], color="black", alpha=0.5)
ax.quiver(0, 0, 0, E[0], E[1], E[2], color="black", alpha=0.5)


# angle parallactique signé
cross = np.cross(t_pole, t_zenith)
sign = np.sign(np.dot(cross, O))

q = sign * np.arccos(np.clip(np.dot(t_pole, t_zenith), -1, 1))

print("q (deg) =", np.degrees(q))
scale = 0.3

ax.quiver(O[0], O[1], O[2], t_pole[0], t_pole[1], t_pole[2], color="red", length=scale)

ax.quiver(O[0], O[1], O[2], t_zenith[0], t_zenith[1], t_zenith[2], color="green", length=scale)


# -----------------------------
# Arc représentant q
# -----------------------------

n_arc = 50
angles = np.linspace(0, q, n_arc)

arc = np.array([np.cos(a) * t_pole + np.sin(a) * t_zenith for a in angles])

# projeter sur la sphère (autour de O)
arc_points = O[None, :] + 0.2 * arc
arc_points /= np.linalg.norm(arc_points, axis=1)[:, None]

ax.plot(arc_points[:, 0], arc_points[:, 1], arc_points[:, 2], color="purple", linewidth=3, label="q")

mid = arc_points[len(arc_points) // 2]
ax.text(mid[0], mid[1], mid[2], "q", color="purple", fontsize=12)

# mise en forme
ax.set_box_aspect([1, 1, 1])
ax.set_xlim([-1, 1])
ax.set_ylim([-1, 1])
ax.set_zlim([-1, 1])

ax.legend()
ax.set_title("Illustration géométrique de l'angle parallactique")

plt.show()

🎯 Résultat visuel attendu

Avec ça, tu verras immédiatement :

🔴 équateur : horizontal (dans le repère équatorial)
🔵 cercle horaire : passe par pôles
🟢 cercle vertical : passe par zénith
🟠 horizon : ⟂ zénith

```
# conventions visuelles fortes
# Pôle céleste → rouge
# Zénith → vert
# Objet → bleu
# Cercle horaire → bleu foncé
# Cercle vertical → vert foncé
# Horizon → orange
# Méridien → noir pointillé
```

Lecture physique de la figure

Avec ces ajouts, tu verras clairement :

✔️ Cercle horaire
passe par les pôles
dépend de H (rotation Terre)
✔️ Cercle vertical
dépend du zénith
géométrie locale
✔️ Méridien
cas particulier du cercle horaire (H = 0)
✔️ Angle parallactique

= angle entre :

direction vers le pôle (dans le plan horaire)
direction vers le zénith (dans le plan vertical)
💡 Point clé (niveau expert)

Ton intuition initiale était bonne :
👉 tu pensais “le cercle horaire ne devrait pas passer par le pôle”

➡️ En réalité :

il DOIT passer par le pôle
sinon ce ne serait pas un cercle horaire

👉 ce qui ne passe PAS par le pôle :

le cercle vertical